# Cox Model

In [ ]:
import pandas as pd

df=pd.read_csv(r"data\cox_moc_final.csv")
df.head()

,NACCID,start_date,event,event_date,last_date,time,NACCAGE,SEX,EDUC,HYPERTEN,...,COGMODE,DEPD,MEMORY,ORIENT,BPDIAS,HYPERTEN_BIN,DIABETES_BIN,HYPERCHO_BIN,CVHATT_BIN,ALCOHOL_BIN
0,NACC000067,2012-05-23,0,NaN,2014-06-25,763,60,1,18,0,...,1,1,0.0,0.0,99.0,0,0,0,0,0
1,NACC000176,2021-02-15,0,NaN,2023-04-17,791,67,2,12,1,...,0,0,0.0,0.0,84.0,1,0,0,0,0
2,NACC000184,2007-01-29,0,NaN,2008-02-20,387,69,2,16,0,...,1,0,0.5,0.5,210.8,0,0,0,0,0
3,NACC000304,2006-04-13,0,NaN,2012-08-22,2323,86,2,18,1,...,1,0,0.0,0.0,71.2,1,0,0,0,0
4,NACC000382,2005-09-27,0,NaN,2010-03-30,1645,72,2,12,1,...,1,1,1.0,0.0,68.0,1,0,0,0,0


In [2]:
df=df.drop(columns=["HYPERTEN","DIABETES","HYPERCHO","CVHATT","ALCOHOL","event_date"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20622 entries, 0 to 20621
Data columns (total 27 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   NACCID        20622 non-null  object 
 1   start_date    20622 non-null  object 
 2   event         20622 non-null  int64  
 3   last_date     20622 non-null  object 
 4   time          20622 non-null  int64  
 5   NACCAGE       20622 non-null  int64  
 6   SEX           20622 non-null  int64  
 7   EDUC          20622 non-null  int64  
 8   STROKE        20622 non-null  int64  
 9   TOBAC30       20622 non-null  int64  
 10  NACCBMI       20622 non-null  float64
 11  NACCFAM       20622 non-null  int64  
 12  NACCALZD      20622 non-null  int64  
 13  VISITYR       20622 non-null  int64  
 14  VISITMO       20622 non-null  int64  
 15  VISITDAY      20622 non-null  int64  
 16  BPSYS         20622 non-null  float64
 17  COGMODE       20622 non-null  int64  
 18  DEPD          20622 non-nu

In [3]:
df['SEX'] = df['SEX'].apply(lambda x: 0 if x == 1 else 1)
print(df['SEX'].value_counts())

df['start_date']= pd.to_datetime(df['start_date'])
df['last_date']= pd.to_datetime(df['last_date'])
df['NACCID']=df['NACCID'].astype(str)
df['time'] = (df['last_date'] - df['start_date']).dt.days

SEX
1    12322
0     8300
Name: count, dtype: int64


In [4]:
cols_to_drop = ['start_date', 'last_date','NACCID','NACCALZD','VISITYR','VISITMO','VISITDAY','BPDIAS']  # datetime 컬럼 리스트
df_for_cox = df.drop(columns=cols_to_drop)

In [5]:
df_for_cox.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20622 entries, 0 to 20621
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   event         20622 non-null  int64  
 1   time          20622 non-null  int64  
 2   NACCAGE       20622 non-null  int64  
 3   SEX           20622 non-null  int64  
 4   EDUC          20622 non-null  int64  
 5   STROKE        20622 non-null  int64  
 6   TOBAC30       20622 non-null  int64  
 7   NACCBMI       20622 non-null  float64
 8   NACCFAM       20622 non-null  int64  
 9   BPSYS         20622 non-null  float64
 10  COGMODE       20622 non-null  int64  
 11  DEPD          20622 non-null  int64  
 12  MEMORY        20622 non-null  float64
 13  ORIENT        20622 non-null  float64
 14  HYPERTEN_BIN  20622 non-null  int64  
 15  DIABETES_BIN  20622 non-null  int64  
 16  HYPERCHO_BIN  20622 non-null  int64  
 17  CVHATT_BIN    20622 non-null  int64  
 18  ALCOHOL_BIN   20622 non-nu

In [6]:
df_for_cox.head()  # Display the first few rows of the DataFrame

,event,time,NACCAGE,SEX,EDUC,STROKE,TOBAC30,NACCBMI,NACCFAM,BPSYS,COGMODE,DEPD,MEMORY,ORIENT,HYPERTEN_BIN,DIABETES_BIN,HYPERCHO_BIN,CVHATT_BIN,ALCOHOL_BIN
0,0,763,60,0,18,0,0,25.50,1,146.0,1,1,0.0,0.0,0,0,0,0,0
1,0,791,67,1,12,0,0,29.70,1,136.0,0,0,0.0,0.0,1,0,0,0,0
2,0,387,69,1,16,0,0,67.14,1,133.0,1,0,0.5,0.5,0,0,0,0,0
3,0,2323,86,1,18,0,0,65.12,0,129.0,1,0,0.0,0.0,1,0,0,0,0
4,0,1645,72,1,12,0,0,31.30,1,112.0,1,1,1.0,0.0,1,0,0,0,0


In [8]:
from lifelines import CoxPHFitter

# cox model 돌릴 데이터프레임: df_cox_clean
# 'time'과 'event' 칼럼이 있어야 함

cph = CoxPHFitter(penalizer=0.1)  # 패널티를 0.1로 설정
cph.fit(df_for_cox, duration_col='time', event_col='event')
cph.print_summary()


<lifelines.CoxPHFitter: fitted with 20622 total observations, 15602 right-censored observations>
             duration col = 'time'
                event col = 'event'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 20622
number of events observed = 5020
   partial log-likelihood = -44567.36
         time fit was run = 2025-07-28 06:53:50 UTC

---
              coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                     
NACCAGE       0.04      1.04      0.00            0.04            0.04                1.04                1.05
SEX          -0.09      0.92      0.02           -0.13           -0.04                0.87                0.96
EDUC         -0.02      0.98      0.00           -0.03           -0.02                0.97                0.98
STROKE       -0.01      0.99      0.07           -0.16            0.13                0.85                1.14
TOBAC30       0.01      1.01      0.06           -0.10            0.13                0.90                1.14
NACCBMI      -0.00      1.00      0.00           -0.01           -0.00                0.99                1.00
NACCFAM       0.12      1.13      0.02            0.08            0.17                1.08                1.19
BPSYS         0.00      1.00      0.00            0.00            0.00                1.00                1.00
COGMODE       0.57      1.77      0.02            0.52            0.61                1.69                1.85
DEPD          0.14      1.15      0.03            0.08            0.20                1.09                1.22
MEMORY        0.59      1.80      0.04            0.52            0.66                1.67                1.94
ORIENT       -0.17      0.85      0.04           -0.25           -0.08                0.78                0.92
HYPERTEN_BIN  0.06      1.06      0.03            0.01            0.11                1.01                1.12
DIABETES_BIN  0.04      1.04      0.04           -0.03            0.11                0.97                1.12
HYPERCHO_BIN  0.05      1.05      0.02            0.00            0.10                1.00                1.10
CVHATT_BIN    0.04      1.04      0.06           -0.07            0.15                0.93                1.16
ALCOHOL_BIN  -0.07      0.93      0.06           -0.19            0.05                0.83                1.05

              cmp to     z      p  -log2(p)
covariate                                  
NACCAGE         0.00 31.80 <0.005    734.74
SEX             0.00 -3.46 <0.005     10.83
EDUC            0.00 -5.97 <0.005     28.68
STROKE          0.00 -0.17   0.87      0.20
TOBAC30         0.00  0.25   0.81      0.31
NACCBMI         0.00 -3.79 <0.005     12.67
NACCFAM         0.00  5.03 <0.005     20.96
BPSYS           0.00  2.38   0.02      5.85
COGMODE         0.00 24.73 <0.005    446.06
DEPD            0.00  4.73 <0.005     18.75
MEMORY          0.00 15.56 <0.005    178.91
ORIENT          0.00 -4.01 <0.005     14.00
HYPERTEN_BIN    0.00  2.42   0.02      6.01
DIABETES_BIN    0.00  1.09   0.27      1.87
HYPERCHO_BIN    0.00  1.97   0.05      4.37
CVHATT_BIN      0.00  0.63   0.53      0.92
ALCOHOL_BIN     0.00 -1.17   0.24      2.05
---
Concordance = 0.76
Partial AIC = 89168.73
log-likelihood ratio test = 2608.07 on 17 df
-log2(p) of ll-ratio test = inf

In [10]:
print(df_for_cox['ORIENT'].value_counts())
print(df_for_cox.groupby('ORIENT')['event'].mean())

print(df_for_cox['MEMORY'].value_counts())
print(df_for_cox.groupby('MEMORY')['event'].mean())

ORIENT
0.0    18074
0.5     1726
1.0      621
2.0      139
3.0       62
Name: count, dtype: int64
ORIENT
0.0    0.227122
0.5    0.415411
1.0    0.272142
2.0    0.172662
3.0    0.080645
Name: event, dtype: float64
MEMORY
0.0    14252
0.5     4749
1.0     1372
2.0      185
3.0       64
Name: count, dtype: int64
MEMORY
0.0    0.182501
0.5    0.399874
1.0    0.346939
2.0    0.200000
3.0    0.109375
Name: event, dtype: float64
